In [ ]:
import os
import gc
import json
import numpy as np
import torch
import pandas as pd
from transformers import AutoModelForCausalLM
import kagglehub

print("=" * 60)
print("NEMOTRON-3 MOE ROUTER COLLAPSE ANALYSIS")
print("Analyzing expert routing weights for specialization vs collapse")
print("=" * 60)

try:
    print("\n[1/4] Loading model structure (meta device)...")
    model_id = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    model_path = kagglehub.model_download(model_id)

    # Load on meta to inspect weights without filling VRAM immediately
    from accelerate import init_empty_weights
    import transformers
    
    # We will load the actual model with device_map="auto" to get the weights
    print("\n[2/4] Loading model weights...")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16
    )
    
    print("\n[3/4] Analyzing router weights across MoE layers...")
    results = []
    
    # Iterate through all layers
    for i, layer in enumerate(model.model.layers):
        # Check if layer has MoE (nemotron uses hybrid, so some layers might not have mlp.gate)
        if hasattr(layer, "mlp") and hasattr(layer.mlp, "gate"):
            gate_weight = layer.mlp.gate.weight.detach().float().cpu().numpy()
            # gate_weight shape: [num_experts, hidden_size]
            num_experts, hidden_size = gate_weight.shape
            
            # Compute variance of weights for each expert
            expert_variances = np.var(gate_weight, axis=1)
            
            # Compute pairwise cosine similarity between experts
            norms = np.linalg.norm(gate_weight, axis=1, keepdims=True)
            normalized_weights = gate_weight / (norms + 1e-8)
            cosine_sim_matrix = np.dot(normalized_weights, normalized_weights.T)
            
            # Exclude self-similarity (diagonal)
            np.fill_diagonal(cosine_sim_matrix, np.nan)
            avg_cosine_sim = np.nanmean(cosine_sim_matrix)
            max_cosine_sim = np.nanmax(cosine_sim_matrix)
            
            # Entropy of the weight magnitudes (proxy for router uniformity)
            magnitudes = np.linalg.norm(gate_weight, axis=1)
            probs = magnitudes / np.sum(magnitudes)
            entropy = -np.sum(probs * np.log(probs + 1e-8))
            
            results.append({
                "layer": i,
                "num_experts": int(num_experts),
                "avg_cosine_sim": float(avg_cosine_sim),
                "max_cosine_sim": float(max_cosine_sim),
                "routing_entropy": float(entropy),
                "min_variance": float(np.min(expert_variances)),
                "max_variance": float(np.max(expert_variances))
            })
            
            print(f"Layer {i} (MoE): Avg Sim={avg_cosine_sim:.4f}, Max Sim={max_cosine_sim:.4f}, Entropy={entropy:.4f}")
        else:
            print(f"Layer {i}: No MoE routing gate found.")

    print("\n[4/4] Saving analysis results...")
    df = pd.DataFrame(results)
    df.to_csv("nemotron_router_analysis.csv", index=False)
    print(df.describe())
    print("\nAnalysis complete! Results saved to nemotron_router_analysis.csv")

except Exception as e:
    import traceback
    print(f"\nERROR: {e}")
    traceback.print_exc()
